# Cohort Creation Workflow

This notebook provides a complete workflow for clearing and rebuilding cohorts.

## Workflow Steps

1. **Configuration**: Set up paths, Python binaries, and cohort parameters
2. **Cleanup**: Clear existing cohort data (S3 and local)
3. **Cohort Creation**: Build cohorts with time-windowed HCG event logic
4. **Verification**: Verify cohorts were created successfully

## Navigation

- **Section A**: Configuration and Setup
- **Section B**: Cleanup Existing Cohorts
- **Section C**: Create Cohorts
- **Section D**: Verify Cohort Creation

---

**Note**: This notebook follows the same structure as the Step 3b Feature Importance EDA workflow notebooks for consistency.

## A. Configuration and Setup

In [ ]:
import sys
import os
from pathlib import Path
import subprocess
import boto3
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set project root
PROJECT_ROOT = Path('/home/pgx3874/pgx-analysis')  # Update for your EC2 path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Jupyter environment Python path (update if different)
PYTHON_BIN = Path('/home/pgx3874/jupyter-env/bin/python3.11')  # Full path to Jupyter Python
if not PYTHON_BIN.exists():
    # Fallback to sys.executable if custom path doesn't exist
    PYTHON_BIN = Path(sys.executable)
    print(f"⚠️  Custom Python path not found, using: {PYTHON_BIN}")

# Import project utilities
from py_helpers.constants import age_band_to_fname, REQUIRED_COHORTS

# Configuration
# Cohorts to create
COHORTS = [
    "opioid_ed",      # OPIOID_ED cohort (F11.20 target)
    "non_opioid_ed"   # POLYPHARMACY COHORT (time-windowed HCG target)
]

# Age bands for each cohort (each cohort has all age bands)
AGE_BANDS = REQUIRED_COHORTS

# Event years to process
EVENT_YEARS = [2016, 2017, 2018, 2019]

# Time window for polypharmacy cohort (non_opioid_ed)
# Options: 7, 14, 21, 30, 45 days (default: 14)
TIME_WINDOW_DAYS = 14

# S3 bucket
S3_BUCKET = "pgxdatalake"
S3_REPO_BUCKET = "pgx-repository"

# Output directories
LOG_DIR = PROJECT_ROOT / "2_create_cohort" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Always use NVMe spill directory for DuckDB
os.environ["DUCKDB_TMP_DIRECTORY"] = "/mnt/nvme/duckdb_tmp"

# Two-job plan: treat total workers as 2 for memory calc
os.environ["PGX_TOTAL_WORKERS"] = "2"

# Optional: enable DuckDB profiling (only if debugging performance)
# os.environ["PGX_ENABLE_DUCKDB_PROFILING"] = "1"


print(f"✅ Configuration loaded")
print(f"   Project Root: {PROJECT_ROOT}")
print(f"   Python Binary: {PYTHON_BIN}")
print(f"   Cohorts: {COHORTS}")
print(f"   Time Window (polypharmacy): {TIME_WINDOW_DAYS} days")
print(f"   Event Years: {EVENT_YEARS}")
print(f"   Log Directory: {LOG_DIR}")

## B. Cleanup Existing Cohorts

**Warning**: This will delete existing cohort data from S3 and local storage.

This step clears:
- Step 2: Cohort parquet files (S3 and local)
- Step 3b: Feature importance outputs
- Step 4a: Model data
- Step 6: Trained models
- Checkpoints (optional)

In [ ]:
# Check what exists before cleanup
s3_client = boto3.client('s3')

print("📊 Checking existing cohort data in S3...")
print("=" * 60)

cohort_paths = [
    f"gold/cohorts/cohort_name={cohort}/" for cohort in COHORTS
]

for cohort_path in cohort_paths:
    try:
        response = s3_client.list_objects_v2(
            Bucket=S3_BUCKET,
            Prefix=cohort_path,
            MaxKeys=10
        )
        
        if 'Contents' in response:
            count = len(response['Contents'])
            total = response.get('KeyCount', count)
            print(f"   {cohort_path}: {count} objects found (showing first 10)")
        else:
            print(f"   {cohort_path}: No objects found")
    except Exception as e:
        print(f"   {cohort_path}: Error checking - {e}")

print("=" * 60)

In [ ]:
# Run cleanup script
# Options:
#   --skip-checkpoints: Keep checkpoints
#   --skip-s3: Only delete local files
#   --skip-local: Only delete S3 files
#   --yes: Auto-confirm (skip interactive prompt)

SKIP_CHECKPOINTS = False  # Set to True to keep checkpoints
SKIP_S3 = False           # Set to True to skip S3 deletion
SKIP_LOCAL = False        # Set to True to skip local deletion
AUTO_CONFIRM = True       # Set to True to skip confirmation prompt (for notebook use)

cleanup_script = PROJECT_ROOT / "utility_scripts" / "cleanup_cohort_data.sh"

if not cleanup_script.exists():
    print(f"❌ Cleanup script not found: {cleanup_script}")
    print("   Skipping cleanup step")
else:
    print("🧹 Running cleanup script...")
    print(f"   Script: {cleanup_script}")
    
    # Build command
    cmd = ["bash", str(cleanup_script)]
    
    if SKIP_CHECKPOINTS:
        cmd.append("--skip-checkpoints")
    if SKIP_S3:
        cmd.append("--skip-s3")
    if SKIP_LOCAL:
        cmd.append("--skip-local")
    if AUTO_CONFIRM:
        cmd.append("--yes")  # Skip confirmation prompt for notebook use
    
    print(f"   Command: {' '.join(cmd)}")
    if AUTO_CONFIRM:
        print(f"   ⚠️  Auto-confirmation enabled (--yes flag)")
    
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), text=True)
    
    if result.returncode == 0:
        print(f"\n✅ Cleanup completed successfully")
    else:
        print(f"\n❌ Cleanup failed with return code {result.returncode}")
        if result.stdout:
            print(f"   Output: {result.stdout[-500:]}")  # Last 500 chars
        if result.stderr:
            print(f"   Error: {result.stderr[-500:]}")  # Last 500 chars

## C. Create Cohorts

This section creates cohorts with the new time-windowed HCG event logic.

Cohort Types:

1. **POLYPHARMACY** (`non_opioid_ed`): Uses time-windowed HCG events as target
2. **OPIOID_ED** (`opioid_ed`): Uses F11.20 as target event (no time windowing)

Time Windows:

For the polypharmacy cohort:
- 21 days (default)
- 0 days filtered out as these are likely discharge medications

The main `is_target_case` column uses the specified `TIME_WINDOW_DAYS`.

### Polypharmacy Cohort

In [ ]:
%%bash
set -euo pipefail

export PGX_DUCKDB_THREADS=16
export PGX_DUCKDB_MEMORY_LIMIT=256GB
export PGX_TOTAL_WORKERS=2

cd /home/pgx3874/pgx-analysis

# Launch ed_non_opioid
nohup /home/pgx3874/jupyter-env/bin/python3.11 2_create_cohort/run_series_ed_non_opioid.py \
  --skip-existing \
  --concurrent-workers 1 \
  > 2_create_cohort/logs/ed_non_opioid_run.log 2>&1 &
echo "ed_non_opioid PID: $!"


### Opioid Cohort

In [ ]:
%%bash
set -euo pipefail

export PGX_DUCKDB_THREADS=12
export PGX_DUCKDB_MEMORY_LIMIT=192GB
export PGX_TOTAL_WORKERS=2

cd /home/pgx3874/pgx-analysis

# Cell 3: Launch opioid_ed
nohup /home/pgx3874/jupyter-env/bin/python3.11 2_create_cohort/run_series_opioid_ed.py \
  --skip-existing \
  --concurrent-workers 1 \
  > 2_create_cohort/logs/opioid_ed_run.log 2>&1 &
echo "opioid_ed PID: $!"


## D. Verify Cohort Creation

This section verifies that cohorts were created successfully by checking S3 and local paths.

In [ ]:
# Verify cohorts exist in S3
s3_client = boto3.client('s3')

print("📊 Verifying cohorts in S3...")
print("=" * 60)

# Note: S3 uses "non_opioid_ed" (not "ed_non_opioid") due to normalize_cohort_name
cohort_mapping = {
    "opioid_ed": "opioid_ed",
    "non_opioid_ed": "non_opioid_ed"  # S3 path uses "non_opioid_ed"
}

all_found = True
for cohort in COHORTS:
    s3_cohort_name = cohort_mapping.get(cohort, cohort)
    
    for age_band in AGE_BANDS.get(cohort, []):
        for event_year in EVENT_YEARS:
            # CORRECT PATH ORDER: cohort_name -> event_year -> age_band
            s3_key = f"gold/cohorts/cohort_name={s3_cohort_name}/event_year={event_year}/age_band={age_band}/cohort.parquet"
            
            try:
                s3_client.head_object(Bucket=S3_BUCKET, Key=s3_key)
                # Get file size
                response = s3_client.head_object(Bucket=S3_BUCKET, Key=s3_key)
                size_mb = response['ContentLength'] / (1024 * 1024)
                print(f"✅ {cohort}/{age_band}/{event_year}: {size_mb:.2f} MB")
            except s3_client.exceptions.ClientError as e:
                if e.response['Error']['Code'] == '404':
                    print(f"❌ {cohort}/{age_band}/{event_year}: NOT FOUND")
                    all_found = False
                else:
                    print(f"⚠️  {cohort}/{age_band}/{event_year}: ERROR - {e}")
                    all_found = False

print("=" * 60)
if all_found:
    print("✅ All cohorts verified in S3")
else:
    print("❌ Some cohorts are missing in S3")

In [ ]:
# Verify multiclass target columns exist for polypharmacy cohort
import duckdb
import os
from pathlib import Path

print("📊 Verifying multiclass target columns for polypharmacy cohort...")
print("=" * 60)

# Resolve local data path using same logic as resolve_local_cohort_root()
def get_local_cohort_path():
    """Resolve local cohort root path (same logic as create_model_data.py)"""
    env_path = os.getenv("LOCAL_DATA_PATH")
    if env_path:
        root = Path(env_path)
        if root.exists():
            return root
    
    # Use get_data_root() logic: /mnt/nvme on Linux
    if os.name != 'nt':  # Linux/EC2
        data_root = Path("/mnt/nvme")
        candidates = [
            data_root / "gold" / "cohorts",  # Linux/EC2: /mnt/nvme/gold/cohorts
            data_root / "data" / "gold_cohorts",  # Alternative Linux path
        ]
        for path in candidates:
            if path.exists():
                return path
        return data_root / "gold" / "cohorts"  # Default to /mnt/nvme/gold/cohorts
    else:
        # Windows/local dev - try common project locations
        import sys
        # Try to find project root from current working directory
        cwd = Path.cwd()
        candidates = [
            cwd / "data" / "gold_cohorts",
            cwd.parent / "data" / "gold_cohorts",
            Path.home() / "pgx_data" / "gold" / "cohorts",
        ]
        for path in candidates:
            if path.exists():
                return path
        # Default fallback
        return cwd / "data" / "gold_cohorts"

local_data_path = get_local_cohort_path()
print(f"Using local data path: {local_data_path}")

# Connect to DuckDB (no S3 setup needed for local files)
con = duckdb.connect()

# Check one polypharmacy cohort file
test_cohort = "non_opioid_ed"  # Local files use "non_opioid_ed" (not "ed_non_opioid")
test_age_band = AGE_BANDS.get("non_opioid_ed", AGE_BANDS.get("ed_non_opioid", []))[0]  # First age band
test_year = EVENT_YEARS[0]  # First year

# Build local file path
local_parquet_path = local_data_path / f"cohort_name={test_cohort}" / f"event_year={test_year}" / f"age_band={test_age_band}" / "cohort.parquet"

print(f"\nChecking: {local_parquet_path}")

if not local_parquet_path.exists():
    print(f"❌ File not found: {local_parquet_path}")
    print("\nTrying alternative cohort name...")
    # Try with "ed_non_opioid" if "non_opioid_ed" doesn't exist
    test_cohort_alt = "ed_non_opioid"
    local_parquet_path = local_data_path / f"cohort_name={test_cohort_alt}" / f"event_year={test_year}" / f"age_band={test_age_band}" / "cohort.parquet"
    if local_parquet_path.exists():
        test_cohort = test_cohort_alt
        print(f"✅ Found with alternative name: {local_parquet_path}")
    else:
        print(f"❌ File not found with alternative name either: {local_parquet_path}")
        con.close()
        exit(1)

try:
    # Get schema from local parquet file
    schema_query = f"DESCRIBE SELECT * FROM read_parquet('{local_parquet_path}') LIMIT 0"
    schema = con.execute(schema_query).fetchdf()
    
    print(f"\nColumns in {test_cohort}/{test_age_band}/{test_year}:")
    
    # Check for multiclass columns
    expected_columns = [
        'is_target_case',
        'is_target_case_7d',
        'is_target_case_14d',
        'is_target_case_21d',
        'is_target_case_30d',
        'is_target_case_45d'
    ]
    
    found_columns = []
    missing_columns = []
    
    for col in expected_columns:
        if col in schema['column_name'].values:
            found_columns.append(col)
            print(f"   ✅ {col}")
        else:
            missing_columns.append(col)
            print(f"   ❌ {col} (missing)")
    
    if len(missing_columns) == 0:
        print("\n✅ All multiclass target columns found")
    else:
        print(f"\n❌ Missing columns: {missing_columns}")
        
except Exception as e:
    print(f"❌ Error checking schema: {e}")
    import traceback
    traceback.print_exc()

con.close()

## Next Steps

After successfully creating cohorts, proceed to:

1. **Step 3b: Feature Importance EDA**
   - Run BupaR post-target analysis
   - Filter post-target leakage features
   - Generate refined feature importance files

2. **Step 4a: Model Data Creation**
   - Create `model_events.parquet` with cases and controls
   - Uses refined features from Step 3b

3. **Step 4b: Event Filtering**
   - Filter administrative codes
   - Remove post-event leakage

4. **Step 5: PGx Feature Engineering**
   - Add PGx drug counts
   - Add drug counts

5. **Step 6: Final Model Training**
   - Train CatBoost, XGBoost, and XGBoost RF models
   - Select best model by recall/AUC-PR

See `WORKFLOW_EXECUTION_TODO.md` for detailed instructions.